# 학습 모델 → 로컬 CPU용 OpenVINO INT8 변환

Colab에서 학습한 `best.pt`를 로컬 PC(GPU 없는 PC)의 CPU에서 빠르게 돌릴 수 있도록 OpenVINO INT8로 변환합니다.
학습이 끝난 뒤 **모델마다 한 번** 실행합니다. 런타임은 CPU여도 됩니다(GPU 불필요).

1. 아래 설정 셀에서 `RUN_NAME`(학습 실행 이름)과 `MODEL_NAME`(저장할 이름)을 정합니다.
2. 위에서부터 차례로 실행합니다.
3. 결과: Drive의 `vm/count_car_ver6.0/models/`에 `<MODEL_NAME>.pt`와 `<MODEL_NAME>_int8_openvino_model/` 폴더가 생깁니다.
4. 로컬 GUI에서 `<MODEL_NAME>.pt`를 고르면, GPU가 없는 PC는 실행 모드 `자동`에서 변환 폴더를 알아서 씁니다.

**이름 주의**: 변환한 뒤에 `.pt` 이름만 바꾸면 짝을 못 찾습니다. 이름은 여기서 정한 대로 두세요.

INT8 보정 이미지는 Drive `yolo_data`의 학습용 ZIP에서 무작위 2,000장만 꺼내 씁니다(압축 전체를 풀지 않음, 라벨 불필요).
ALZip `.egg` 파일은 일부만 꺼낼 수 없어 건너뜁니다.

In [ ]:
!pip -q install -U ultralytics openvino nncf

In [ ]:
import importlib.util, subprocess, sys
# 위 설치 셀을 건너뛰었어도 동작하도록 필요한 패키지가 없으면 여기서 설치한다.
if not all(importlib.util.find_spec(m) for m in ('ultralytics', 'openvino', 'nncf')):
    print('필요한 패키지를 설치합니다...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'ultralytics', 'openvino', 'nncf'], check=True)

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# ===== 설정 =====
RUN_NAME = 'YOLO26n_final_1280'   # MyDrive/yolo_runs/ 아래 학습 실행 이름
MODEL_NAME = 'yolo26n_v1'         # 로컬 models/ 에 저장할 이름 (영문·숫자·_ 권장)
IMGSZ = 1280                      # 학습 imgsz (긴 변)
ASPECT = (16, 9)                  # 현장 영상 가로:세로. 정사각형이면 None
N_CALIB = 2000                    # 보정 이미지 수
SEED = 42
# ================

DRIVE_ROOT = Path('/content/drive/MyDrive')
BEST_PT = DRIVE_ROOT / 'yolo_runs' / RUN_NAME / 'weights' / 'best.pt'
ZIP_DIR = DRIVE_ROOT / 'yolo_data'
MODELS_DIR = DRIVE_ROOT / 'vm' / 'count_car_ver6.0' / 'models'
WORK = Path('/content/export_work')
CALIB = Path('/content/calib')

assert BEST_PT.is_file(), f'학습 결과가 없습니다: {BEST_PT}'
assert MODELS_DIR.is_dir(), f'프로젝트 models 폴더가 없습니다: {MODELS_DIR}'
target = MODELS_DIR / f'{MODEL_NAME}.pt'
if target.exists():
    print(f'주의: {target.name} 이 이미 있습니다. 덮어씁니다.')
print('모델:', BEST_PT)

In [ ]:
import random, zipfile, shutil

if CALIB.exists():
    shutil.rmtree(CALIB)
(CALIB / 'images' / 'val').mkdir(parents=True)
(CALIB / 'labels' / 'val').mkdir(parents=True)

members = []
for zp in sorted(ZIP_DIR.glob('batch_*.zip')):
    with zipfile.ZipFile(zp) as zf:
        members += [(zp, n) for n in zf.namelist() if n.lower().endswith('.jpg')]
print(f'ZIP {len(list(ZIP_DIR.glob("batch_*.zip")))}개, 이미지 {len(members):,}장 중 {N_CALIB}장 사용')
random.Random(SEED).shuffle(members)
picked = members[:N_CALIB]

by_zip = {}
for zp, name in picked:
    by_zip.setdefault(zp, []).append(name)
for zp, names in by_zip.items():
    with zipfile.ZipFile(zp) as zf:
        all_names = set(zf.namelist())
        for name in names:
            stem = Path(name).name
            (CALIB / 'images' / 'val' / stem).write_bytes(zf.read(name))
            label = name[:-4] + '.txt'
            if label in all_names:
                (CALIB / 'labels' / 'val' / (stem[:-4] + '.txt')).write_bytes(zf.read(label))
    print(f'  {zp.name}: {len(names)}장')

names_yaml = ''
for zp in sorted(ZIP_DIR.glob('batch_*.zip')):
    with zipfile.ZipFile(zp) as zf:
        cls = [n for n in zf.namelist() if n.endswith('classes.txt')]
        if cls:
            names_yaml = ''.join(f'  {i}: {c.strip()}\n' for i, c in enumerate(zf.read(cls[0]).decode('utf-8-sig').splitlines()) if c.strip())
            break
assert names_yaml, 'classes.txt 를 찾지 못했습니다.'
calib_yaml = CALIB / 'calib.yaml'
calib_yaml.write_text(f'path: {CALIB}\ntrain: images/val\nval: images/val\nnames:\n{names_yaml}', encoding='utf-8')
print(calib_yaml.read_text())

In [ ]:
import math
from ultralytics import YOLO

if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
local_pt = WORK / f'{MODEL_NAME}.pt'
shutil.copy2(BEST_PT, local_pt)

if ASPECT:
    w, h = ASPECT
    short = math.ceil(IMGSZ * min(w, h) / max(w, h) / 32) * 32
    shape = [short, IMGSZ] if w >= h else [IMGSZ, short]
else:
    shape = [IMGSZ, IMGSZ]
print('입력 크기(세로x가로):', shape)

ov_dir = Path(YOLO(str(local_pt)).export(format='openvino', imgsz=shape, quantize=8, data=str(calib_yaml)))
print('변환 완료:', ov_dir)

## INT8 정확도 점검

보정에 쓴 것과 다른 이미지가 좋지만, 여기서는 빠르게 보정 이미지 일부로 원본 `.pt`와 INT8 결과를 비교합니다.
- **일치율**: 원본이 찾은 차량 중 INT8도 같은 위치(IoU 0.5 이상)에서 찾은 비율
- 95% 이상이면 대체로 문제없습니다. 크게 낮으면 원본 `.pt`를 GPU로 쓰거나 FP32 변환을 검토하세요.

In [ ]:
import numpy as np
import cv2

def iou(a, b):
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda r: (r[:, 2] - r[:, 0]) * (r[:, 3] - r[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

pt_model = YOLO(str(local_pt))
ov_model = YOLO(str(ov_dir), task='detect')
images = sorted((CALIB / 'images' / 'val').glob('*.jpg'))[:100]
total = matched = same_cls = ov_total = 0
for p in images:
    img = cv2.imread(str(p))
    a = pt_model.predict(img, imgsz=IMGSZ, rect=True, conf=0.25, verbose=False, device='cpu')[0].boxes
    b = ov_model.predict(img, imgsz=IMGSZ, conf=0.25, verbose=False)[0].boxes
    total += len(a); ov_total += len(b)
    if len(a) and len(b):
        m = iou(a.xyxy.numpy(), b.xyxy.numpy())
        best = m.argmax(1); hit = m.max(1) >= 0.5
        matched += int(hit.sum())
        same_cls += int((hit & (a.cls.numpy() == b.cls.numpy()[best])).sum())
print(f'이미지 {len(images)}장 | 원본 차량 {total}대, INT8 {ov_total}대')
print(f'위치 일치율 {100 * matched / max(1, total):.1f}% | 차종까지 일치 {100 * same_cls / max(1, total):.1f}%')

In [ ]:
dst_pt = MODELS_DIR / f'{MODEL_NAME}.pt'
dst_ov = MODELS_DIR / ov_dir.name
shutil.copy2(local_pt, dst_pt)
if dst_ov.exists():
    shutil.rmtree(dst_ov)
shutil.copytree(ov_dir, dst_ov)
print('저장 완료')
print(' ', dst_pt)
print(' ', dst_ov)
print('로컬 GUI 에서', dst_pt.name, '을 고르면 GPU 없는 PC 는 자동으로 INT8 모델을 씁니다.')